NLP internship -- week 1 -- start

In [ ]:
!pip install pandas
!pip install nltk
!pip install scikit-learn
!pip install torchtext
import torch
%pip install model
!pip install matplotlib


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Loading the data

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv", nrows=10000)
# Keep only what we need
df = df[["comment_text", "toxic"]]

df.head()

1. Stack Query and Image columns (vstack)

2. Remove duplicates

In [ ]:
stacked_df = df.rename(columns={
    "comment_text": "text",
    "toxic": "label"
})

stacked_df.head()

Dont forget to add weighting as they are still unbalanced

Now do the Classical NLP steps :
1- tokenize 
2- remove stop words
3- stemming or lemmatization

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

texts = stacked_df['text'].values
corpus = []
keep_mask = []   # tracks which rows survive tokenization

for i in range(len(texts)):
    review = re.sub('[^a-zA-Z]', ' ', str(texts[i]))
    review = review.lower()
    review = review.split()
    review = [lemmatizer.lemmatize(word) for word in review if word not in stop_words]

    if len(review) > 0:
        corpus.append(review)
        keep_mask.append(True)
    else:
        keep_mask.append(False)

# drop the same rows from stacked_df so text/label alignment is preserved
dropped = len(keep_mask) - sum(keep_mask)
print(f"Dropped {dropped} empty documents out of {len(keep_mask)}")

stacked_df = stacked_df[keep_mask].reset_index(drop=True)

tokenized_docs = corpus
print(tokenized_docs[:3])

Now Start numbering each unique word 


Get unique word to index and index to word

In [ ]:
all_words = []

for doc in corpus:
    for word in doc:
        all_words.append(word)

unique_words = set(all_words)

print(len(unique_words))

In [ ]:
word_to_idx = {}

word_to_idx['<PAD>'] = 0
word_to_idx['<UNK>'] = 1

idx = 2
for word in unique_words:
    word_to_idx[word] = idx
    idx += 1

print(len(word_to_idx))

In [ ]:
idx_to_word = {}

for word, index in word_to_idx.items():
    idx_to_word[index] = word

print(idx_to_word[0])
print(idx_to_word[1])

In [ ]:
print(word_to_idx['<PAD>'])   # should print 0
print(word_to_idx['<UNK>'])   # should print 1
print(len(word_to_idx) == len(idx_to_word))   # should be True

# pick a random real word and confirm both directions agree
sample_word = list(word_to_idx.keys())[5]
sample_idx = word_to_idx[sample_word]
print(idx_to_word[sample_idx] == sample_word)   # should be True

Now Actually convert the numbers into indeces

get maximum length in order to add paddings 

In [ ]:
doc_lengths = []
for doc in corpus:
    doc_lengths.append(len(doc))

import numpy as np
doc_lengths = np.array(doc_lengths)

print(doc_lengths.min())
print(doc_lengths.max())
print(doc_lengths.mean())
print(np.percentile(doc_lengths, 95))

In [ ]:
encoded_docs = []

for doc in corpus:
    encoded_doc = []
    for word in doc:
        if word in word_to_idx:
            encoded_doc.append(word_to_idx[word])
        else:
            encoded_doc.append(word_to_idx['<UNK>'])
    encoded_docs.append(encoded_doc)

print(encoded_docs[:2])

In [ ]:
padded_docs = []

max_len= 120
for seq in encoded_docs:
    if len(seq) < max_len:
        seq = seq + [word_to_idx['<PAD>']] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    padded_docs.append(seq)

print(padded_docs[:2])
print(len(padded_docs[0]), len(padded_docs[1]))  # should both equal max_len

In [ ]:
seq_lengths = []
for doc in corpus:
    seq_lengths.append(min(len(doc), max_len))

seq_lengths = np.array(seq_lengths)

In [ ]:
print((doc_lengths > 30).sum())
print(np.sort(doc_lengths)[-5:])  # the 5 longest documents

Now i should encode the labels and add weightings based on this 
Toxic Category
Safe                  882
Violent Crimes        701
unsafe                230
Non-Violent Crimes    208

In [ ]:
label_to_idx = {0: 'Not Toxic', 1: 'Toxic'}

encoded_labels = stacked_df['label'].values
print(label_to_idx)

In [ ]:
idx_to_label = label_to_idx

Creating an embedding layer

In [ ]:
from torchtext.vocab import GloVe

glove = GloVe(
    name='6B',
    dim=100 # this is the embedding dimension of the GloVe vectors. You can choose 50, 100, 200, or 300 

)

all_unique_words_size = len(word_to_idx)
embedding_dim = 100

# creating the embdeding matrix
import torch

embedding_matrix = torch.zeros(all_unique_words_size, embedding_dim)

for word, idx in word_to_idx.items():

    if word in glove.stoi:
        embedding_matrix[idx] = glove[word]

# finished creating the embedding matrix, now we can use it in our model
import torch.nn as nn

embedding = nn.Embedding.from_pretrained(
    embedding_matrix,
    freeze=False # it is false means the vectors can change 
)




In [ ]:
covered = sum(1 for w in word_to_idx if w in glove.stoi)
total = len(word_to_idx)
print(f"Coverage: {covered}/{total} = {covered/total:.2%}")


minority_labels = [1]
minority_indices = stacked_df[stacked_df['label'].isin(minority_labels)].index

minority_words = set()
for idx in minority_indices:
    for w in corpus[idx]:
        minority_words.add(w)

oov_words = [w for w in minority_words if w not in glove.stoi]
print(f"OOV rate on cleaned tokens: {len(oov_words)}/{len(minority_words)} = {len(oov_words)/len(minority_words):.2%}")
print(f"Sample OOV words: {oov_words[:20]}")

Splitting data into training data and test data and validation data

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

X = np.array(padded_docs)
y = np.array(encoded_labels)
lengths = seq_lengths

X_train, X_temp, y_train, y_temp, len_train, len_temp = train_test_split(
    X, y, lengths, test_size=0.2, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test, len_val, len_test = train_test_split(
    X_temp, y_temp, len_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(X_train.shape, X_val.shape, X_test.shape)
print(np.bincount(y_train))
print(np.bincount(y_val))
print(np.bincount(y_test))

adding weighting

In [ ]:
class_counts = np.bincount(y_train)
total_samples = len(y_train)
num_classes = 2

class_weights = {}
for i in range(num_classes):
    class_weights[i] = total_samples / (num_classes * class_counts[i])

print(class_weights)

Convert to tensors and creating data loader

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

X_train_tensor = torch.LongTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
len_train_tensor = torch.LongTensor(len_train)

X_val_tensor = torch.LongTensor(X_val)
y_val_tensor = torch.LongTensor(y_val)
len_val_tensor = torch.LongTensor(len_val)

X_test_tensor = torch.LongTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)
len_test_tensor = torch.LongTensor(len_test)

train_dataset = TensorDataset(X_train_tensor, len_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, len_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, len_test_tensor, y_test_tensor)

# weight per training sample = inverse frequency of its class
sample_weights = [class_weights[label] for label in y_train]
sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=256, sampler=sampler,
    num_workers=2
)
val_loader = DataLoader(
    val_dataset, batch_size=256, shuffle=False,
    num_workers=2
)
test_loader = DataLoader(
    test_dataset, batch_size=256, shuffle=False,
    num_workers=2
)

Defining my RNN model

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

class LSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim, num_classes, dropout=0.4):
        super(LSTMClassifier, self).__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=True,
            padding_idx=word_to_idx['<PAD>']
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)   # *2 for bidirectional, same as before

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.lstm(packed)   # <-- note the extra unpacking here

        # hidden shape: (num_directions, batch, hidden_dim)
        hidden_cat = torch.cat((hidden[0], hidden[1]), dim=1)

        last_hidden = self.dropout(hidden_cat)
        logits = self.fc(last_hidden)
        return logits

Training the RNN 

Set up device, model, loss, optimizer


In [ ]:
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_classes = 2

## you can change the hidden dim
model = LSTMClassifier(embedding_matrix, hidden_dim=64, num_classes=num_classes).to(device)
# turn your class_weights dict into a tensor, ordered by class index 0..3
weights_tensor = torch.tensor([class_weights[i] for i in range(num_classes)], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss()

## you can change the lr
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

The training loop skeleton


In [ ]:
from sklearn.metrics import f1_score
import torch

num_epochs = 8
best_val_f1 = 0.0
patience = 5
epochs_no_improve = 0

for epoch in range(num_epochs):

    # Unfreeze embeddings after 5 epochs
    if epoch == 5:
        model.embedding.weight.requires_grad = True
        print("Embedding layer unfrozen.")

    ############################
    # Training
    ############################
    model.train()

    total_loss = 0.0

    for x_batch, len_batch, y_batch in train_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(x_batch, len_batch)

        loss = criterion(logits, y_batch)

        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    ############################
    # Validation
    ############################
    model.eval()

    val_loss = 0.0
    val_preds = []
    val_true = []

    with torch.no_grad():

        for x_batch, len_batch, y_batch in val_loader:

            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch, len_batch)

            loss = criterion(logits, y_batch)

            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_true.extend(y_batch.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    val_f1 = f1_score(
        val_true,
        val_preds,
        average="macro"
    )

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

    ############################
    # Early Stopping
    ############################
    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        epochs_no_improve = 0

        torch.save(model.state_dict(), "best_lstm_model.pt")

        print("✓ Best model saved.")

    else:

        epochs_no_improve += 1

        print(
            f"No improvement ({epochs_no_improve}/{patience})"
        )

        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nBest Validation Macro F1: {best_val_f1:.4f}")

2. Add the test-set evaluation cell


In [ ]:
from sklearn.metrics import classification_report

model.load_state_dict(torch.load('best_lstm_model.pt'))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x_batch, len_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        logits = model(x_batch, len_batch)
        preds = torch.argmax(logits, dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(y_batch.tolist())

target_names = [idx_to_label[i] for i in range(num_classes)]
print(classification_report(all_labels, all_preds, target_names=target_names))

ROC curve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import torch.nn.functional as F

model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for x_batch, len_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        logits = model(x_batch, len_batch)
        probs = F.softmax(logits, dim=1).cpu().numpy()   # convert logits -> probabilities

        all_probs.append(probs)
        all_labels.extend(y_batch.tolist())

all_probs = np.vstack(all_probs)          # shape: (n_samples, num_classes)
all_labels = np.array(all_labels)

# one-hot encode true labels for one-vs-rest ROC
y_test_bin = label_binarize(all_labels, classes=list(range(num_classes)))

# compute ROC curve and AUC for each class
fpr = {}
tpr = {}
roc_auc = {}

for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], all_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# micro-average ROC curve (aggregates all classes' TP/FP counts together)
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), all_probs.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# plot
plt.figure(figsize=(8, 6))

for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label=f'{idx_to_label[i]} (AUC = {roc_auc[i]:.2f})')

plt.plot(fpr["micro"], tpr["micro"], linestyle='--', color='black',
         label=f'micro-average (AUC = {roc_auc["micro"]:.2f})')

plt.plot([0, 1], [0, 1], linestyle=':', color='gray')  # diagonal = random guessing baseline
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — One-vs-Rest per Class')
plt.legend(loc='lower right')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(all_labels, all_preds)
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print(cm_df)